In [ ]:
# Block 0 - Imports and Setup for secondary calculations

import pandas as pd
import numpy as np
import statsmodels.api as sm

# ── LOAD TRADES ───────────────────────────────────────────────────────────────
trades = pd.read_csv('Trading_Data_with_Jurisdiction.csv')
trades['Traded'] = pd.to_datetime(trades['Traded'])

# Lock down the exact sample period
trades = trades[(trades['Traded'] >= '2013-01-01') & (trades['Traded'] <= '2025-12-31')].copy()

# Drop exchanges
trades = trades[~trades['Transaction'].str.contains('Exchange', case=False, na=False)].copy()

# Normalize and drop Other
def normalise_type(val):
    v = str(val).strip().lower()
    if 'purchase' in v: 
        return 'Buy'
    elif 'sale' in v:   
        return 'Sell'
    return 'Other'

trades['Type'] = trades['Transaction'].apply(normalise_type)
trades = trades[trades['Type'] != 'Other'].copy()
# ── FREQUENT / INFREQUENT CLASSIFICATION ─────────────────────────────────────

trades = pd.read_csv('Trading_Data_with_Jurisdiction.csv')
trades['Traded'] = pd.to_datetime(trades['Traded'])
trades['Party']   = trades['Party'].str.strip().str.upper()
trades['Chamber'] = trades['Chamber'].str.strip().str.capitalize()
trades['Type']    = trades['Transaction'].apply(normalise_type)

trade_counts  = trades.groupby('BioGuideID').size().reset_index(name='Total_Trades')

q25 = trade_counts['Total_Trades'].quantile(0.25)  # ~4 trades
q75 = trade_counts['Total_Trades'].quantile(0.75)  # ~174 trades

print(f"Total members:    {len(trade_counts):,}")
print(f"Q25 threshold:    {q25:.0f} trades")
print(f"Q75 threshold:    {q75:.0f} trades")

# Top quartile = Frequent, bottom quartile = Infrequent, middle 50% dropped
trade_counts['Trader_Type'] = np.where(
    trade_counts['Total_Trades'] >= q75, 'Frequent',
    np.where(trade_counts['Total_Trades'] <= q25, 'Infrequent', 'Middle')
)

print(f"\nMember-level split:")
print(trade_counts['Trader_Type'].value_counts())

# Only keep top and bottom quartile members in the analysis
trades = trades.merge(
    trade_counts[['BioGuideID', 'Total_Trades', 'Trader_Type']],
    on='BioGuideID', how='left'
)

print(f"\nTrade-level split (Middle excluded from portfolios):")
print(trades['Trader_Type'].value_counts())

# ── LOAD DAILY STOCK RETURNS (wide → long) ────────────────────────────────────
print("\nLoading daily stock returns...")
ret_wide = pd.read_csv('congress_stock_returns_daily.csv')

# A1 = 'Date', B1 onwards = tickers
ret_wide = ret_wide.rename(columns={ret_wide.columns[0]: 'Date'})
ret_wide['Date'] = pd.to_datetime(ret_wide['Date'], format='%Y-%m-%d')

# Melt to long format: Date | Ticker | Daily_Return
ret_long = ret_wide.melt(id_vars='Date', var_name='Ticker', value_name='Daily_Return')
ret_long['Daily_Return'] = pd.to_numeric(ret_long['Daily_Return'], errors='coerce')
ret_long = ret_long.dropna(subset=['Daily_Return'])

print(f"Daily return observations: {len(ret_long):,}")
print(f"Tickers covered: {ret_long['Ticker'].nunique():,}")
print(f"Date range: {ret_long['Date'].min().date()} to {ret_long['Date'].max().date()}")

# ── LOAD DAILY FAMA-FRENCH FACTORS ───────────────────────────────────────────
ff_raw = pd.read_csv(
    'F-F_Research_Data_Factors_daily.csv',
    skiprows=4,
    header=0,
    names=['Date', 'MktRF', 'SMB', 'HML', 'RF']
)
ff_raw = ff_raw[pd.to_numeric(ff_raw['MktRF'], errors='coerce').notna()].copy()
ff_raw['Date'] = pd.to_datetime(ff_raw['Date'].astype(str).str.strip(), format='%Y%m%d')
for col in ['MktRF', 'SMB', 'HML', 'RF']:
    ff_raw[col] = pd.to_numeric(ff_raw[col], errors='coerce') / 100

mom_raw = pd.read_csv(
    'F-F_Momentum_Factor_daily.csv',
    skiprows=13,
    header=0,
    names=['Date', 'UMD']
)
mom_raw = mom_raw[pd.to_numeric(mom_raw['UMD'], errors='coerce').notna()].copy()
mom_raw['Date'] = pd.to_datetime(mom_raw['Date'].astype(str).str.strip(), format='%Y%m%d')
mom_raw['UMD']  = pd.to_numeric(mom_raw['UMD'], errors='coerce') / 100

ff_daily = ff_raw.merge(mom_raw, on='Date', how='inner')
print(f"\nDaily factors: {len(ff_daily):,} days "
      f"({ff_daily['Date'].min().date()} to {ff_daily['Date'].max().date()})")

# ── HELPER: DAILY CALENDAR-TIME PORTFOLIO ────────────────────────────────────
def build_calendar_portfolio(subset_df, holding_days):
    """
    For each trading day, computes the equal-weighted return of all stocks
    whose trades are currently active (within the holding window).
    Returns a daily time series of portfolio excess returns.
    """
    subset_df = subset_df.dropna(subset=['Traded']).copy()

    if len(subset_df) < 10:
        return pd.DataFrame(columns=['Date','Port_Ret','N','MktRF','SMB','HML','UMD','RF','Excess_Ret'])

    # Assign exit date for each trade
    subset_df['Exit_Date'] = subset_df['Traded'] + pd.Timedelta(days=holding_days)

    # Merge trades with daily returns on Ticker
    merged = subset_df[['Ticker', 'Traded', 'Exit_Date']].merge(
        ret_long, on='Ticker', how='inner'
    )

    # Keep only days where the trade is active
    merged = merged[
        (merged['Date'] >= merged['Traded']) &
        (merged['Date'] <= merged['Exit_Date'])
    ]

    if len(merged) == 0:
        return pd.DataFrame(columns=['Date','Port_Ret','N','MktRF','SMB','HML','UMD','RF','Excess_Ret'])

    # Equal-weighted daily portfolio return
    daily_port = (
        merged.groupby('Date')
        .agg(Port_Ret=('Daily_Return', 'mean'), N=('Daily_Return', 'count'))
        .reset_index()
    )

    # Merge with FF factors
    daily_port = daily_port.merge(ff_daily, on='Date', how='inner')
    daily_port['Excess_Ret'] = daily_port['Port_Ret'] - daily_port['RF']

    return daily_port.dropna(subset=['Excess_Ret', 'MktRF'])


# ── HELPER: CARHART REGRESSION (DAILY) ───────────────────────────────────────
def run_carhart(port_df, label):
    """
    Runs Carhart four-factor regression on daily excess portfolio returns.
    Alpha is annualised by multiplying by 252.
    HAC standard errors with 21 lags (approx. one trading month).
    """
    if len(port_df) < 60:
        print(f"  WARNING: {label} — only {len(port_df)} days, skipping")
        return {
            'Portfolio': label, 'Alpha_Ann': np.nan, 'Alpha_tstat': np.nan,
            'Mkt_Beta': np.nan, 'Mkt_tstat': np.nan,
            'SMB': np.nan, 'HML': np.nan, 'UMD': np.nan,
            'Adj_R2': np.nan, 'N_obs': len(port_df)
        }

    y = port_df['Excess_Ret']
    X = sm.add_constant(port_df[['MktRF', 'SMB', 'HML', 'UMD']])
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 21})

    return {
        'Portfolio':   label,
        'Alpha_Ann':   round(model.params['const'] * 252 * 100, 2),
        'Alpha_tstat': round(model.tvalues['const'], 2),
        'Mkt_Beta':    round(model.params['MktRF'], 2),
        'Mkt_tstat':   round(model.tvalues['MktRF'], 2),
        'SMB':         round(model.params['SMB'], 2),
        'HML':         round(model.params['HML'], 2),
        'UMD':         round(model.params['UMD'], 2),
        'Adj_R2':      round(model.rsquared_adj, 3),
        'N_obs':       int(model.nobs)
    }


# ── HELPER: RUN ONE SUBGROUP SPLIT ───────────────────────────────────────────
def run_split(trades_df, group_col, group_vals, horizon):
    results = []
    h = horizon
    for gval in group_vals:
        sub = trades_df[trades_df[group_col] == gval].copy()
        print(f"\n  {gval} (n={len(sub):,})")

        portfolios = {
            f'{gval}_Jur_Buy_{h}d':     sub[(sub['Jurisdiction_Trade']==1) & (sub['Type']=='Buy')],
            f'{gval}_Jur_Sell_{h}d':    sub[(sub['Jurisdiction_Trade']==1) & (sub['Type']=='Sell')],
            f'{gval}_NonJur_Buy_{h}d':  sub[(sub['Jurisdiction_Trade']==0) & (sub['Type']=='Buy')],
            f'{gval}_NonJur_Sell_{h}d': sub[(sub['Jurisdiction_Trade']==0) & (sub['Type']=='Sell')],
        }
        for name, df in portfolios.items():
            print(f"    {name}: {len(df):,} trades")
            port = build_calendar_portfolio(df, h)
            results.append(run_carhart(port, name))

        # 1. Build the base portfolios for the spreads
        p_jur_buy = build_calendar_portfolio(sub[(sub['Jurisdiction_Trade']==1) & (sub['Type']=='Buy')], h)
        p_nj_buy  = build_calendar_portfolio(sub[(sub['Jurisdiction_Trade']==0) & (sub['Type']=='Buy')], h)
        
        p_jur_sell = build_calendar_portfolio(sub[(sub['Jurisdiction_Trade']==1) & (sub['Type']=='Sell')], h)
        p_nj_sell  = build_calendar_portfolio(sub[(sub['Jurisdiction_Trade']==0) & (sub['Type']=='Sell')], h)

        # 2. Buy Spread: Long Jur, Short Non-Jur
        if not p_jur_buy.empty and not p_nj_buy.empty:
            spread_buy = p_jur_buy.merge(p_nj_buy, on='Date', suffixes=('_j','_n'))
            spread_buy['Excess_Ret'] = spread_buy['Excess_Ret_j'] - spread_buy['Excess_Ret_n']
            for fac in ['MktRF','SMB','HML','UMD']: spread_buy[fac] = spread_buy[f'{fac}_j']
            results.append(run_carhart(spread_buy, f'{gval}_Spread_Buy_{h}d'))

        # 3. Sell Spread: Long Non-Jur, Short Jur (Matches thesis text)
        if not p_jur_sell.empty and not p_nj_sell.empty:
            spread_sell = p_jur_sell.merge(p_nj_sell, on='Date', suffixes=('_j','_n'))
            spread_sell['Excess_Ret'] = spread_sell['Excess_Ret_n'] - spread_sell['Excess_Ret_j']
            for fac in ['MktRF','SMB','HML','UMD']: spread_sell[fac] = spread_sell[f'{fac}_j']
            results.append(run_carhart(spread_sell, f'{gval}_Spread_Sell_{h}d'))

    return results


# ── HELPER: FORMAT RESULTS ────────────────────────────────────────────────────
def format_results(results_list):
    df = pd.DataFrame(results_list)
    def stars(t):
        if pd.isna(t): return ''
        if abs(t) >= 2.576: return '***'
        if abs(t) >= 1.960: return '**'
        if abs(t) >= 1.645: return '*'
        return ''
    df['Alpha_Ann_%'] = df.apply(
        lambda r: f"{r['Alpha_Ann']}{stars(r['Alpha_tstat'])}"
        if pd.notna(r['Alpha_Ann']) else 'N/A', axis=1
    )
    return df

# ── FINAL DATA SUMMARY FOR METHODOLOGY SECTION ───────────────────────────────
n_total = len(trades)
n_buys  = (trades['Type'] == 'Buy').sum()
n_sells = (trades['Type'] == 'Sell').sum()
n_jur   = (trades['Jurisdiction_Trade'] == 1).sum()
n_non_jur = (trades['Jurisdiction_Trade'] == 0).sum()

print("── FINAL CLEANED SAMPLE (2013-2025) ──")
print(f"Total trades analyzed (N):    {n_total:,}")
print(f"  - Buy transactions:         {n_buys:,}")
print(f"  - Sell transactions:        {n_sells:,}")
print(f"  - Jurisdictional trades:    {n_jur:,}")
print(f"  - Non-jurisdictional:       {n_non_jur:,}")

# ── SANITY CHECK ─────────────────────────────────────────────────────────────
print("\nSanity check:")
print(f"  Jur Buy:     {((trades['Jurisdiction_Trade']==1) & (trades['Type']=='Buy')).sum():,}")
print(f"  Jur Sell:    {((trades['Jurisdiction_Trade']==1) & (trades['Type']=='Sell')).sum():,}")
print(f"  NonJur Buy:  {((trades['Jurisdiction_Trade']==0) & (trades['Type']=='Buy')).sum():,}")
print(f"  NonJur Sell: {((trades['Jurisdiction_Trade']==0) & (trades['Type']=='Sell')).sum():,}")

print("\nBlock 0 complete. Run Blocks 1, 2, and 3.")

In [ ]:
# Block 1 - Baseline Analysis

print("=" * 60)
print("ANALYSIS 1: BASELINE FULL SAMPLE")
print("=" * 60)

results_baseline = []
# Section 4.1 evaluates 20, 130, and 255 day horizons
horizons = [20, 130, 255] 

for h in horizons:
    print(f"\n--- {h}-day horizon ---")

    # 1. Base Portfolios
    portfolios = {
        f'Full_Jur_Buy_{h}d':     trades[(trades['Jurisdiction_Trade']==1) & (trades['Type']=='Buy')],
        f'Full_Jur_Sell_{h}d':    trades[(trades['Jurisdiction_Trade']==1) & (trades['Type']=='Sell')],
        f'Full_NonJur_Buy_{h}d':  trades[(trades['Jurisdiction_Trade']==0) & (trades['Type']=='Buy')],
        f'Full_NonJur_Sell_{h}d': trades[(trades['Jurisdiction_Trade']==0) & (trades['Type']=='Sell')],
    }

    for name, df in portfolios.items():
        print(f"    {name}: {len(df):,} trades")
        port = build_calendar_portfolio(df, h)
        results_baseline.append(run_carhart(port, name))

    # 2. Spread Portfolios (Matching your paper's exact definitions)
    p_jur_buy = build_calendar_portfolio(trades[(trades['Jurisdiction_Trade']==1) & (trades['Type']=='Buy')], h)
    p_nj_buy  = build_calendar_portfolio(trades[(trades['Jurisdiction_Trade']==0) & (trades['Type']=='Buy')], h)
    
    p_jur_sell = build_calendar_portfolio(trades[(trades['Jurisdiction_Trade']==1) & (trades['Type']=='Sell')], h)
    p_nj_sell  = build_calendar_portfolio(trades[(trades['Jurisdiction_Trade']==0) & (trades['Type']=='Sell')], h)

    # Buy Spread: Long Jur, Short Non-Jur
    if not p_jur_buy.empty and not p_nj_buy.empty:
        spread_buy = p_jur_buy.merge(p_nj_buy, on='Date', suffixes=('_j','_n'))
        spread_buy['Excess_Ret'] = spread_buy['Excess_Ret_j'] - spread_buy['Excess_Ret_n']
        for fac in ['MktRF','SMB','HML','UMD']: spread_buy[fac] = spread_buy[f'{fac}_j']
        results_baseline.append(run_carhart(spread_buy, f'Full_Spread_Buy_{h}d'))

    # Sell Spread: Long Non-Jur, Short Jur
    if not p_jur_sell.empty and not p_nj_sell.empty:
        spread_sell = p_jur_sell.merge(p_nj_sell, on='Date', suffixes=('_j','_n'))
        spread_sell['Excess_Ret'] = spread_sell['Excess_Ret_n'] - spread_sell['Excess_Ret_j']
        for fac in ['MktRF','SMB','HML','UMD']: spread_sell[fac] = spread_sell[f'{fac}_j']
        results_baseline.append(run_carhart(spread_sell, f'Full_Spread_Sell_{h}d'))

df_baseline = format_results(results_baseline)
df_baseline.to_csv('Results_Baseline.csv', index=False)

cols = ['Portfolio','Alpha_Ann_%','Alpha_tstat','Mkt_Beta','SMB','HML','UMD','Adj_R2','N_obs']
print("\n=== BASELINE RESULTS ===")
print(df_baseline[cols].to_string(index=False))
print("\nSaved: Results_Baseline.csv")

In [ ]:
# Block 2 - House vs Senate analysis

print("=" * 60)
print("ANALYSIS 1: HOUSE vs SENATE")
print("=" * 60)

results_chamber = []
for h in [20, 130, 255]:
    print(f"\n--- {h}-day horizon ---")
    results_chamber += run_split(trades, 'Chamber', ['House', 'Senate'], h)

df_chamber = format_results(results_chamber)
df_chamber.to_csv('Results_Chamber.csv', index=False)

cols = ['Portfolio','Alpha_Ann_%','Alpha_tstat','Mkt_Beta','SMB','HML','UMD','Adj_R2','N_obs']
print("\n=== CHAMBER RESULTS ===")
print(df_chamber[cols].to_string(index=False))
print("\nSaved: Results_Chamber.csv")

In [ ]:
# Block 3 - Dem vs Rep Analysis

print("=" * 60)
print("ANALYSIS 2: DEMOCRATS vs REPUBLICANS")
print("=" * 60)

results_party = []
for h in [20, 130, 255]:
    print(f"\n--- {h}-day horizon ---")
    results_party += run_split(trades, 'Party', ['D', 'R'], h)

df_party = format_results(results_party)
df_party.to_csv('Results_Party.csv', index=False)

cols = ['Portfolio','Alpha_Ann_%','Alpha_tstat','Mkt_Beta','SMB','HML','UMD','Adj_R2','N_obs']
print("\n=== PARTY RESULTS ===")
print(df_party[cols].to_string(index=False))
print("\nSaved: Results_Party.csv")

In [ ]:
# Block 4 - Frequent trade analysis (Quartile-Based)

# Check if trades exists before running to avoid the NameError
if 'trades' not in locals():
    print("CRITICAL ERROR: 'trades' dataframe not found. Please run Block 0 first.")
else:
    # 1. Filter out the "Middle" group to isolate Frequent vs. Infrequent
    # Using the 'Trader_Type' column created in Block 0 merge
    trades_freq_analysis = trades[trades['Trader_Type'] != 'Middle'].copy()

    print("=" * 60)
    print("ANALYSIS 3: FREQUENT vs INFREQUENT TRADERS")
    print("=" * 60)

    # These variables (q75, q25) are defined in your Block 0
    print(f"Frequent Threshold (Q75):   >= {q75:.0f} trades")
    print(f"Infrequent Threshold (Q25): <= {q25:.0f} trades")
    print("Note: The middle 50% of members are excluded from this analysis.")

    print("\nTrade-level distribution in analysis sample:")
    print(trades_freq_analysis['Trader_Type'].value_counts().to_string())

    results_freq = []
    for h in [20, 130, 255]:
        print(f"\n--- {h}-day horizon ---")
        # Ensure run_split is defined in your helper functions
        results_freq += run_split(trades_freq_analysis, 'Trader_Type', ['Frequent', 'Infrequent'], h)

    df_freq = format_results(results_freq)

    # 2. Apply the "Not Estimable" mask for sparse portfolios (< 60 observations)
    # This prevents interpreting noise as an informational signal
    sparse_mask = df_freq['N_obs'] < 60
    df_freq.loc[sparse_mask, ['Alpha_Ann_%', 'Alpha_tstat']] = "Not Estimable"

    # 3. Output Results
    df_freq.to_csv('Results_TraderFrequency.csv', index=False)

    cols = ['Portfolio', 'Alpha_Ann_%', 'Alpha_tstat', 'Mkt_Beta', 'SMB', 'HML', 'UMD', 'Adj_R2', 'N_obs']
    print("\n=== TRADER FREQUENCY RESULTS ===")
    print(df_freq[cols].to_string(index=False))
    print("\nSaved: Results_TraderFrequency.csv")

In [ ]:
# Block 4 - Committee-Specific Analysis
import pandas as pd

print("── STARTING COMMITTEE-SPECIFIC ANALYSIS ──")

# 1. LOAD ADDITIONAL FILES
assignments = pd.read_csv('Committee_Assignments_Panel.csv')
matrix = pd.read_csv('Jurisdictional_Matrix_Final.csv')

# 2. PREPARE THE MATRIX (Explode semicolon-separated industries)
matrix.columns = matrix.columns.str.strip()
matrix['ind_list'] = matrix['Mapped_Industries'].str.split(';')
matrix_exploded = matrix.explode('ind_list')
matrix_exploded['ind_list'] = matrix_exploded['ind_list'].str.strip()

# 3. ALIGN SESSIONS
def get_congress(dt):
    year = dt.year
    if year < 2013: return 112
    return 112 + (year - 2011) // 2

# We use the 'trades' dataframe already cleaned by Block 0
trades['Congress_Session'] = trades['Traded'].apply(get_congress)

# 4. MERGE TO FIND RESPONSIBLE COMMITTEES
# A. Get member's committees at the time of trade
trade_assignments = trades.merge(
    assignments[['BioGuideID', 'Congress_Session', 'Committee_Code']], 
    on=['BioGuideID', 'Congress_Session'], 
    how='inner'
)

# B. Match trade's 'Industry' with matrix's 'ind_list'
full_map = trade_assignments.merge(
    matrix_exploded[['Committee_Code', 'Parent_Committee_Name', 'ind_list']], 
    left_on=['Committee_Code', 'Industry'],
    right_on=['Committee_Code', 'ind_list'],
    how='inner'
)

# 5. DEFINE THE TARGET COMMITTEES
target_committees = [
    "House Committee on Armed Services",
    "House Committee on Financial Services",
    "House Committee on Energy and Commerce",
    "Senate Committee on Banking, Housing, and Urban Affairs" 
]

# FAILSAFE: Ensure 'Type' exists regardless of Block 0's notebook state
if 'Type' not in full_map.columns:
    print("  [Auto-Fix] 'Type' column missing. Regenerating from 'Transaction'...")
    def hard_normalise(val):
        v = str(val).strip().lower()
        if 'purchase' in v: return 'Buy'
        elif 'sale' in v:   return 'Sell'
        return 'Other'
    full_map['Type'] = full_map['Transaction'].apply(hard_normalise)

committee_results = []
horizon = 20 

for comm in target_committees:
    print(f"\nProcessing {comm}...")
    
    comm_df = full_map[full_map['Parent_Committee_Name'] == comm].copy()
    
    # Filter using the guaranteed 'Type' column
    comm_sells = comm_df[comm_df['Type'] == 'Sell'].copy()
    comm_buys  = comm_df[comm_df['Type'] == 'Buy'].copy()
    
    print(f"  Found {len(comm_sells)} Sells and {len(comm_buys)} Buys.")
    
    # Sells Regression
    if len(comm_sells) >= 10:
        port_sell = build_calendar_portfolio(comm_sells, horizon)
        if not port_sell.empty:
            committee_results.append(run_carhart(port_sell, f"{comm}_Sell_{horizon}d"))
    else:
        print(f"  Not enough sells to run regression for {comm}")
    
    # Buys Regression 
    if len(comm_buys) >= 10:
        port_buy = build_calendar_portfolio(comm_buys, horizon)
        if not port_buy.empty:
            committee_results.append(run_carhart(port_buy, f"{comm}_Buy_{horizon}d"))

# 6. FORMAT AND DISPLAY RESULTS
print("\n── COMMITTEE-SPECIFIC REGRESSION RESULTS ──")
if committee_results:
    df_comm_results = format_results(committee_results)
    display_cols = ['Portfolio', 'Alpha_Ann', 'Alpha_tstat', 'Alpha_Ann_%', 'Adj_R2', 'N_obs']
    print(df_comm_results[display_cols].to_string(index=False))
    df_comm_results.to_csv('Results_Committees_20d.csv', index=False)
else:
    print("No valid portfolios generated. Check N-counts.")

In [ ]:
# Block 5 - Robustness Checks for HFS 
import pandas as pd
import numpy as np
import statsmodels.api as sm

print("── STARTING ROBUSTNESS CHECKS (HFS 20-DAY SELLS) ──")

# 1. Isolate the target portfolio
hfs_sells = full_map[(full_map['Parent_Committee_Name'] == 'House Committee on Financial Services') & 
                     (full_map['Type'] == 'Sell')].copy()

robustness_results = []

# --- TEST 1: SUB-PERIOD STABILITY ---
# Did this only work during the COVID crash, or is it a persistent structural edge?
hfs_early = hfs_sells[hfs_sells['Traded'] < '2020-01-01'].copy()
hfs_late  = hfs_sells[hfs_sells['Traded'] >= '2020-01-01'].copy()

port_early = build_calendar_portfolio(hfs_early, 20)
port_late  = build_calendar_portfolio(hfs_late, 20)

if not port_early.empty: robustness_results.append(run_carhart(port_early, "HFS_Sell_20d_Pre2020"))
if not port_late.empty:  robustness_results.append(run_carhart(port_late, "HFS_Sell_20d_Post2020"))


# --- TEST 2: ALTERNATIVE HOLDING PERIODS ---
# Prove that the "Velocity of Information" peaks short-term and fades.
test_horizons = [5, 10, 40]
for h in test_horizons:
    port_alt = build_calendar_portfolio(hfs_sells, h)
    if not port_alt.empty:
        robustness_results.append(run_carhart(port_alt, f"HFS_Sell_{h}d_Horizon"))


# --- TEST 3: OUTLIER WINSORIZATION ---
# Prove the alpha isn't driven by a single stock dropping 80% in one day.
port_base_20d = build_calendar_portfolio(hfs_sells, 20)

if not port_base_20d.empty:
    # Calculate the 1st and 99th percentiles of the portfolio's excess return
    lower_bound = port_base_20d['Excess_Ret'].quantile(0.01)
    upper_bound = port_base_20d['Excess_Ret'].quantile(0.99)
    
    # Clip (Winsorize) the extreme daily returns
    port_winsorized = port_base_20d.copy()
    port_winsorized['Excess_Ret'] = port_winsorized['Excess_Ret'].clip(lower=lower_bound, upper=upper_bound)
    
    robustness_results.append(run_carhart(port_winsorized, "HFS_Sell_20d_Winsorized_1%"))

# --- DISPLAY RESULTS ---
print("\n── ROBUSTNESS CHECK RESULTS ──")
if robustness_results:
    df_robust = format_results(robustness_results)
    display_cols = ['Portfolio', 'Alpha_Ann', 'Alpha_tstat', 'Alpha_Ann_%', 'Adj_R2', 'N_obs']
    print(df_robust[display_cols].to_string(index=False))
    df_robust.to_csv('Results_Robustness_HFS.csv', index=False)
else:
    print("Insufficient data to run robustness checks.")

In [ ]:
# Block 6 - Macro Robustness Checks for the Overall Thesis
import pandas as pd
import numpy as np
import statsmodels.api as sm

print("── STARTING MACRO ROBUSTNESS CHECKS (ALL JURISDICTIONAL SELLS) ──")

# 1. Isolate the core overarching portfolio: All Jurisdictional Sells
# Using the main 'trades' dataframe from Block 0
jur_sells = trades[(trades['Jurisdiction_Trade'] == 1) & (trades['Type'] == 'Sell')].copy()

macro_robustness = []
horizon = 20

# Get the baseline portfolio for manipulations
port_base = build_calendar_portfolio(jur_sells, horizon)

if not port_base.empty:
    # --- TEST 1: SUB-PERIOD STABILITY ---
    jur_sells_early = jur_sells[jur_sells['Traded'] < '2020-01-01'].copy()
    jur_sells_late  = jur_sells[jur_sells['Traded'] >= '2020-01-01'].copy()
    
    port_early = build_calendar_portfolio(jur_sells_early, horizon)
    port_late  = build_calendar_portfolio(jur_sells_late, horizon)
    
    if not port_early.empty: macro_robustness.append(run_carhart(port_early, "All_Jur_Sell_Pre2020"))
    if not port_late.empty:  macro_robustness.append(run_carhart(port_late, "All_Jur_Sell_Post2020"))

    # --- TEST 2: OUTLIER WINSORIZATION (1% and 99%) ---
    lower_bound = port_base['Excess_Ret'].quantile(0.01)
    upper_bound = port_base['Excess_Ret'].quantile(0.99)
    
    port_win = port_base.copy()
    port_win['Excess_Ret'] = port_win['Excess_Ret'].clip(lower=lower_bound, upper=upper_bound)
    
    macro_robustness.append(run_carhart(port_win, "All_Jur_Sell_Winsorized_1%"))

    # --- TEST 3: ALTERNATIVE ASSET PRICING (CAPM) ---
    # Running a 1-factor model instead of 4-factor
    y_capm = port_base['Excess_Ret']
    X_capm = sm.add_constant(port_base[['MktRF']])
    model_capm = sm.OLS(y_capm, X_capm).fit(cov_type='HAC', cov_kwds={'maxlags': 21})
    
    alpha_ann_capm = (model_capm.params['const'] * 252) * 100
    tstat_capm = model_capm.tvalues['const']
    
    macro_robustness.append({
        'Portfolio': "All_Jur_Sell_CAPM_Only",
        'Alpha_Ann': round(alpha_ann_capm, 2),
        'Alpha_tstat': round(tstat_capm, 2),
        'Mkt_Beta': round(model_capm.params['MktRF'], 2),
        'Mkt_tstat': round(model_capm.tvalues['MktRF'], 2),
        'SMB': np.nan, 'HML': np.nan, 'UMD': np.nan,
        'Adj_R2': round(model_capm.rsquared_adj, 3),
        'N_obs': int(model_capm.nobs)
    })

# --- DISPLAY RESULTS ---
print("\n── MACRO ROBUSTNESS CHECK RESULTS ──")
if macro_robustness:
    df_macro_robust = format_results(macro_robustness)
    display_cols = ['Portfolio', 'Alpha_Ann', 'Alpha_tstat', 'Alpha_Ann_%', 'Adj_R2', 'N_obs']
    print(df_macro_robust[display_cols].to_string(index=False))
    df_macro_robust.to_csv('Results_Macro_Robustness.csv', index=False)
else:
    print("Insufficient data to run macro robustness checks.")